# D12 — Kernel-Verified Detector & Readout Metrology (Technical)

Companion to `papers/D12/paper_draft.tex`.

Every quantity below is proved in Lean 4; the Python re-computation is a drift check on
`src/core/formulas.py`, not independent evidence.

**Two-layer honesty.** The mathematics is kernel-verified. The identification of any symbol with a
physical instrument — what counts as an absorbed photon, which plane a power is referred to — is the
consuming application's declared hypothesis and never enters the theorems.

In [1]:
import sys, pathlib
if str(pathlib.Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np
from src.core import formulas as F
from src.core import visualizations as V
print("substrate imported")

substrate imported


## 1. The Le Cam / Bhattacharyya floor (Phase 6EA)

`poisson_avgError_floor` — for **every** randomized count rule δ : ℕ → [0,1]:

$$P_e \;\ge\; \tfrac14 \exp\!\big(-(\sqrt{N_a}-\sqrt{N_b})^2\big)$$

No false-alarm constraint, no monotone-likelihood-ratio assumption, no threshold structure.

In [2]:
print("  N_b   N_a     Le Cam floor    Bhattacharyya coeff")
for nb, na in ((0.0, 1.0), (5.0, 10.0), (50.0, 60.0)):
    print(f" {nb:>4}  {na:>4}   {F.poisson_avg_error_floor(na, nb):>12.6e}   {F.poisson_bhattacharyya_coefficient(na, nb):>12.6e}")

# poisson_avgError_equalRates_eq_half: at coincident rates the error is EXACTLY 1/2
print(f"\ncoincident rates -> floor = {F.poisson_avg_error_floor(7.0, 7.0):g}  (Le Cam constant gives 1/4)")
print("Lean poisson_avgError_equalRates_eq_half proves the TRUE value is exactly 1/2,")
print("which is what measures the factor-2 slack in the Le Cam constant.")

  N_b   N_a     Le Cam floor    Bhattacharyya coeff
  0.0   1.0   9.196986e-02   6.065307e-01
  5.0  10.0   1.060167e-01   6.512041e-01
 50.0  60.0   1.585345e-01   7.963279e-01

coincident rates -> floor = 0.25  (Le Cam constant gives 1/4)
Lean poisson_avgError_equalRates_eq_half proves the TRUE value is exactly 1/2,
which is what measures the factor-2 slack in the Le Cam constant.


## 2. The folklore floor fails in **two** directions

The folklore rule of thumb is `miss ≥ exp(−(N_a − N_b))`. Lean refutes it twice, in senses that are
genuinely distinct.

### (A) False-strict as a miss bound — `folklore_miss_floor_false`

The ideal unit-threshold counter (admissible, by `isCountRule_thresholdRule`) misses with
probability exactly `exp(−N_a)`, undershooting the folklore value by the factor `exp(N_b)`, for
**every** bright baseline. `folklore_missFloor_beaten_148fold` certifies a factor of 148 at (5, 10).

In [3]:
nb, na = 5.0, 10.0
actual   = F.poisson_dark_baseline_miss_optimum(na)     # exp(-N_a), the realizable counter
folklore = F.folklore_miss_floor(na, nb)                # exp(-(N_a - N_b))
print(f"realizable unit-threshold miss  exp(-N_a)      = {actual:.6e}")
print(f"folklore value                  exp(-(Na-Nb))  = {folklore:.6e}")
print(f"ratio                                          = {folklore/actual:.4f}   [= e^N_b = {np.exp(nb):.4f}]")
print(f"Lean certifies a factor of at least 148        : {folklore/actual >= 148}")
assert actual < folklore
print("\nOK - the folklore value is NOT a lower bound on miss probability.")

realizable unit-threshold miss  exp(-N_a)      = 4.539993e-05
folklore value                  exp(-(Na-Nb))  = 6.737947e-03
ratio                                          = 148.4132   [= e^N_b = 148.4132]
Lean certifies a factor of at least 148        : True

OK - the folklore value is NOT a lower bound on miss probability.


### (B) Exponentially fail-open as an average-error screen — `folklore_avgFloor_unsound_of_bright`

`folkloreGap_split` is an identity:

$$(N_a - N_b) - (\sqrt{N_a}-\sqrt{N_b})^2 \;=\; 2\sqrt{N_b}\,(\sqrt{N_a}-\sqrt{N_b})$$

Whenever that gap exceeds `log 4`, the **true** floor strictly **exceeds** the folklore value — so a
screen built on the folklore form admits configurations the true floor forbids.

In [4]:
# verify the split identity is exact
worst = 0.0
for nb in (1.0, 5.0, 50.0):
    for na in np.linspace(nb, nb + 40, 300):
        lhs = (na - nb) - (np.sqrt(na) - np.sqrt(nb))**2
        worst = max(worst, abs(lhs - F.folklore_gap_exponent(na, nb)))
print(f"max deviation from folkloreGap_split identity : {worst:.3e}")
assert worst < 1e-9

nb, na = 50.0, 60.0
gap  = F.folklore_gap_exponent(na, nb)
true = F.poisson_avg_error_floor(na, nb)
folk = F.folklore_miss_floor(na, nb)
print(f"\ngap exponent at (50, 60)  = {gap:.5f}    [Lean brightGap_5060: > 9.54]")
print(f"log 4 fail-open threshold = {np.log(4):.5f}")
print(f"true Le Cam floor         = {true:.6e}")
print(f"folklore value            = {folk:.6e}")
print(f"ratio true/folklore       = {true/folk:.2f}    [Lean certifies > 1000]")
assert gap > np.log(4) and true > folk and true/folk > 1000
print("\nOK - fail-open confirmed; the folklore screen admits what the true floor forbids.")

max deviation from folkloreGap_split identity : 2.132e-14

gap exponent at (50, 60)  = 9.54451    [Lean brightGap_5060: > 9.54]
log 4 fail-open threshold = 1.38629
true Le Cam floor         = 1.585345e-01
folklore value            = 4.539993e-05
ratio true/folklore       = 3491.96    [Lean certifies > 1000]

OK - fail-open confirmed; the folklore screen admits what the true floor forbids.


In [5]:
# viz-ref: fig_d12_poisson_floor_vs_folklore
V.fig_d12_poisson_floor_vs_folklore().show()

## 3. Gaussian tail sandwich and the ENBW realizability floor (6EA W2 / 6EB W1)

`gaussianTail_chernoff` / `gaussianTail_mills` bound Q(z) above; `gaussianTail_birnbaum` bounds it
below at **every** real z. `enbw_mul_window_ge_half` is the realizability floor, and
`enbw_mul_window_isLeast` proves 1/2 is the *least* element — sharp, attained **iff** the filter is a
positive multiple of the boxcar (`enbw_eq_half_iff_boxcar`).

In [6]:
print("   z      Birnbaum(lo)        Q(z)        Chernoff(hi)      Mills(hi)")
for z in (0.5, 1.0, 2.0, 3.0):
    q  = F.gaussian_q(z)
    lo = F.gaussian_tail_birnbaum_lower(z)
    ch = F.gaussian_tail_chernoff_upper(z)
    mi = F.gaussian_tail_mills_upper(z)
    assert lo <= q <= min(ch, mi) + 1e-15
    print(f" {z:>4}   {lo:>12.6e}  {q:>12.6e}  {ch:>12.6e}  {mi:>12.6e}")
print("\nOK - the sandwich holds at every sampled z.")

print(f"\nENBW*T, matched boxcar : {F.enbw_boxcar(2.0)*2.0:g}   [Lean floor, attained: 1/2]")
print(f"ENBW*T, DC-matched ramp: {F.enbw_ramp(2.0)*2.0:.6f}   [Lean enbw_ramp_gt_half: > 1/2]")
assert abs(F.enbw_boxcar(2.0)*2.0 - 0.5) < 1e-12 and F.enbw_ramp(2.0)*2.0 > 0.5
print("OK - the floor is saturated by the boxcar and is a real screen for the ramp.")

   z      Birnbaum(lo)        Q(z)        Chernoff(hi)      Mills(hi)
  0.5   1.408261e-01  3.085375e-01  4.412485e-01  7.041307e-01
  1.0   1.209854e-01  1.586553e-01  3.032653e-01  2.419707e-01
  2.0   2.159639e-02  2.275013e-02  6.766764e-02  2.699548e-02
  3.0   1.329555e-03  1.349898e-03  5.554498e-03  1.477283e-03

OK - the sandwich holds at every sampled z.

ENBW*T, matched boxcar : 0.5   [Lean floor, attained: 1/2]
ENBW*T, DC-matched ramp: 0.666667   [Lean enbw_ramp_gt_half: > 1/2]
OK - the floor is saturated by the boxcar and is a real screen for the ramp.


In [7]:
# viz-ref: fig_d12_enbw_matched_filter
V.fig_d12_enbw_matched_filter().show()

## 4. Electrothermal feedback (Phase 6EC)

`etf_stable_iff` is an **exact dichotomy**: every perturbation decays iff `ℒ > −1`. Two natural
criteria are kernel-refuted — `magnitudeOnly_criterion_unsound` (replacing dR/dT by its magnitude
flips an unstable device to stable) and `absLoopGain_criterion_unsound` (`|ℒ|` does not determine
stability).

The Johnson correction is `|1 − ℒ|`, **not** `|1 + ℒ|`. An earlier treatment modelled Johnson noise
as pure *output* noise, omitting a first-order ETF effect from the very layer whose subject is
electrothermal feedback. At ℒ = 3 an ETF-unaware budget is low by exactly 2, not 4.

In [8]:
# absLoopGain_criterion_unsound: same |L|, opposite sides of the boundary
for L in (2.0, -2.0):
    print(f"  L = {L:>5}  |L| = {abs(L):g}   stable = {F.etf_is_stable(L)}")
assert F.etf_is_stable(2.0) and not F.etf_is_stable(-2.0)
print("  -> |L| alone does NOT determine stability.\n")

# tau_eff is NEGATIVE on the unstable branch - reading it through |.| inverts the physics
for L in (3.0, -0.5, -2.0):
    print(f"  L = {L:>5}  tau_eff = {F.etf_effective_time_constant(1.0, 1.0, L):>8.4f}  stable = {F.etf_is_stable(L)}")

print("\n  Johnson NEP correction |1 - L| (equality, not a direction):")
for L in (0.0, 1.0, 1.5, 3.0, 5.0, -3.0):
    f = F.johnson_nep_correction_factor(L)
    verdict = "naive OVERSTATES" if f < 1 else ("exact" if f == 1 else "naive understates")
    print(f"    L = {L:>5}  |1-L| = {f:>4g}   {verdict}")
assert F.johnson_nep_correction_factor(3.0) == 2.0            # NOT 4
assert F.johnson_nep_correction_factor(5.0) == F.johnson_nep_correction_factor(-3.0)   # stability-blind
print("\nOK - factor is 2 at L=3 (not 4), and is stability-blind (L=5 and L=-3 both give 4).")

  L =   2.0  |L| = 2   stable = True
  L =  -2.0  |L| = 2   stable = False
  -> |L| alone does NOT determine stability.

  L =   3.0  tau_eff =   0.2500  stable = True
  L =  -0.5  tau_eff =   2.0000  stable = True
  L =  -2.0  tau_eff =  -1.0000  stable = False

  Johnson NEP correction |1 - L| (equality, not a direction):
    L =   0.0  |1-L| =    1   exact
    L =   1.0  |1-L| =    0   naive OVERSTATES
    L =   1.5  |1-L| =  0.5   naive OVERSTATES
    L =   3.0  |1-L| =    2   naive understates
    L =   5.0  |1-L| =    4   naive understates
    L =  -3.0  |1-L| =    4   naive understates

OK - factor is 2 at L=3 (not 4), and is stability-blind (L=5 and L=-3 both give 4).


In [9]:
# viz-ref: fig_d12_etf_stability
V.fig_d12_etf_stability().show()

## 5. Scope — what is *not* claimed

- **Our Chernoff bound is a SHARPENING, not a first.** Mathlib already kernel-checks a sub-Gaussian
  Chernoff bound at our pin; ours is a factor of 2 tighter.
- **The CPTP / Kraus / Choi / POVM stack is already formalized** in Lean PhysLib, Coq CoqQ and
  Isabelle. D12 claims none of it.
- **Quantum hypothesis testing already exists** in PhysLib, and D12 does **not** consume it — a
  Poisson experiment on ℕ is not a finite type, and an asymmetric Neyman–Pearson rate is a different
  object from a symmetric Bayes/Le Cam floor.
- **Two prior-art checks remain open and are blocking**: an out-of-Mathlib Lean hypothesis-testing
  project, and the Isabelle AFP / Coq `infotheo` libraries.
- **Two of five mechanism ceilings have only degenerate biting witnesses** (zero matched budget —
  a signal-free readout).

Eleven tracked Props carry modelling content. The two most substantial:
`IsWhiteFilteredVariance` (Parseval + spectral flatness live in the *definition*) and
`IsShotFilteredMoments` (Campbell's theorem lives in the *definition*).